# Create UGC Bangladesh ICSETEP Awards

Creates OpenAlex award rows from the University Grants Commission of Bangladesh's official ICSETEP Research and Development Grant approved-subprojects PDF.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/ugc_bd_icsetep_to_s3.py` to download the current official ICSETEP page/PDF, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** Official ICSETEP R&D Grant page at `https://icsetep.ugc.gov.bd/research-and-development-grant/`, which links to `https://icsetep.ugc.gov.bd/wp-content/uploads/2026/01/ASP-Details.pdf` (`List of Approved Sub-Projects from Round 1`).

**Source scope:** ICSETEP Research and Development Grant Round 1 approved sub-projects only. Local full `--skip-upload` validation on 2026-05-27 parsed 20 rows with 100% title, focus area, principal investigator, and affiliation coverage. The official PDF does not publish per-subproject amounts, so `amount` and `currency` are intentionally NULL with a section 6.7 waiver.

**S3 input:** `s3a://openalex-ingest/awards/ugc_bd_icsetep/ugc_bd_icsetep_projects.parquet`

**Funder:** University Grants Commission of Bangladesh, `F4320316035`, country `BD`, DOI `10.13039/100015747`, no ROR in the OpenAlex API response.

**Provenance:** `ugc_bd_icsetep_rdg`


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ugc_bd_icsetep_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ugc_bd_icsetep/ugc_bd_icsetep_projects.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 20 rows.
SELECT COUNT(*) as total_ugc_bd_icsetep_projects
FROM openalex.awards.ugc_bd_icsetep_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.ugc_bd_icsetep_raw;


In [ ]:
%sql
-- Sample raw UGC Bangladesh ICSETEP data.
SELECT
    funder_award_id,
    display_name,
    principal_investigator,
    pi_affiliation,
    area,
    area_label,
    source_year,
    landing_page_url,
    source_pdf_url
FROM openalex.awards.ugc_bd_icsetep_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

The official approved-subprojects PDF does not publish per-row amount or currency. This is an explicit section 6.7 waiver, following the existing no-amount direct-source precedents for NOMIS projects, Wenner-Gren grantees, Fritz Thyssen fundings, HHMI, CIFAR, and Damon Runyon fellowship-pattern ingests.


In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Uses information_schema, not DESCRIBE as a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'ugc_bd_icsetep_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency|bdt|taka';


In [ ]:
%sql
-- Native ID uniqueness check. Should be 20 total and 20 distinct in the full run.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    SUM(CASE WHEN funder_award_id IS NULL OR TRIM(funder_award_id) = '' THEN 1 ELSE 0 END) AS missing_funder_award_id
FROM openalex.awards.ugc_bd_icsetep_raw;


In [ ]:
%sql
-- Required source-field coverage.
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS rows_with_title,
    COUNT(principal_investigator) AS rows_with_pi,
    COUNT(pi_affiliation) AS rows_with_affiliation,
    COUNT(area) AS rows_with_area,
    COUNT(area_label) AS rows_with_area_label,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(principal_investigator), COUNT(*)) * 100.0, 1) AS pct_pi,
    ROUND(try_divide(COUNT(pi_affiliation), COUNT(*)) * 100.0, 1) AS pct_affiliation
FROM openalex.awards.ugc_bd_icsetep_raw;


In [ ]:
%sql
-- Program focus-area distribution from the source PDF.
SELECT area, area_label, COUNT(*) AS rows
FROM openalex.awards.ugc_bd_icsetep_raw
GROUP BY area, area_label
ORDER BY area;


In [ ]:
%sql
-- Amount/currency are intentionally NULL because the official source does not publish per-subproject values.
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    COUNT(currency) AS rows_with_currency,
    collect_set(currency) AS currencies
FROM openalex.awards.ugc_bd_icsetep_raw;


## Step 1.6: Funder Existence Fail-Fast

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE funder_id = 4320316035`. If this returns zero rows, STOP; the insert would silently emit no awards.


In [ ]:
%sql
-- Must return exactly 1 row for University Grants Commission of Bangladesh.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320316035;


In [ ]:
%sql
-- Fails the cell if the Crossref-registered funder row is missing from openalex.common.funder.
SELECT CASE
    WHEN COUNT(*) = 1 THEN 'OK: UGC Bangladesh funder row exists'
    ELSE raise_error('UGC Bangladesh funder row missing from openalex.common.funder')
END AS funder_check
FROM openalex.common.funder
WHERE funder_id = 4320316035;


## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ugc_bd_icsetep_awards
USING delta
AS
WITH
ugc_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320316035
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(source_year AS INT) AS parsed_start_year
    FROM openalex.awards.ugc_bd_icsetep_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,
        TRIM(g.display_name) as display_name,
        NULLIF(TRIM(g.description), '') as description,
        f.funder_id,
        g.native_award_id as funder_award_id,
        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        COALESCE(NULLIF(TRIM(g.funding_type), ''), 'research') as funding_type,
        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), NULLIF(TRIM(g.area_label), ''), 'ICSETEP Research and Development Grant - Round 1') as funder_scheme,
        'ugc_bd_icsetep_rdg' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_start_year as start_year,
        CAST(NULL AS INT) as end_year,
        struct(
            NULLIF(TRIM(g.pi_given_name), '') as given_name,
            NULLIF(TRIM(g.pi_family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            CAST(NULL AS DATE) as role_start,
            struct(
                NULLIF(TRIM(g.pi_affiliation), '') as name,
                'BD' as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               CAST(abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS STRING)) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN ugc_funder f
)
SELECT *
FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 158

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ugc_bd_icsetep_rdg' AND priority = 158;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    158 as priority
FROM openalex.awards.ugc_bd_icsetep_awards;


## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_ugc_bd_icsetep_awards
FROM openalex.awards.ugc_bd_icsetep_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.ugc_bd_icsetep_awards;


In [ ]:
%sql
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    lead_investigator.given_name AS pi_given_name,
    lead_investigator.family_name AS pi_family_name,
    lead_investigator.affiliation.name AS affiliation,
    landing_page_url
FROM openalex.awards.ugc_bd_icsetep_awards
LIMIT 10;


In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT id) AS distinct_openalex_ids, COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.ugc_bd_icsetep_awards;


In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.ugc_bd_icsetep_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(currency) as has_currency,
    COUNT(start_year) as has_start_year,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.family_name) as has_pi_family,
    COUNT(lead_investigator.affiliation.name) as has_affiliation,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) as pct_affiliation
FROM openalex.awards.ugc_bd_icsetep_awards;


In [ ]:
%sql
-- Section 6.7 amount check is intentionally waived for this source.
-- The official approved-list PDF does not publish per-subproject amounts or a per-subproject currency.
SELECT
    'WAIVED: official ICSETEP approved-subprojects PDF has no per-row amount/currency; keep NULLs rather than infer from program-level materials.' AS amount_waiver_note,
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount
FROM openalex.awards.ugc_bd_icsetep_awards;


In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) AS rows
FROM openalex.awards.ugc_bd_icsetep_awards
GROUP BY funder_scheme, funding_type
ORDER BY rows DESC;


In [ ]:
%sql
-- Verify the combined raw table received exactly this source/priority after Step 3.
SELECT provenance, priority, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ugc_bd_icsetep_rdg'
  AND priority = 158
GROUP BY provenance, priority;


In [ ]:
%sql
-- Native-code uniqueness check; must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.ugc_bd_icsetep_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


## Handoff / Admin Notes

- Contractor local validation produced 20 rows from the official ICSETEP approved-subprojects PDF and wrote string-typed parquet locally.
- Contractor has no AWS/S3/Databricks access. Admin must run `scripts/local/ugc_bd_icsetep_to_s3.py` without `--skip-upload`, run this notebook on Databricks, inspect Step 6, and then run the combined `CreateAwards.ipynb` after QA.
- Do not add job YAML until the admin upload, notebook run, and QA are complete.
